# Web 前端
## 功能
- 显示相似度和特征匹配点
- 可以调节检索结果的数量
- 可以显示检索耗时
- RANSAC 阈值可调
- 点击结果图：如果有关键点，会显示关键点；没有关键点，会显示结果图大图
- 匹配点
- 原图和搜索到的图的匹配点之间可以连线，不过有的挨得太近或者重叠了，所以视觉上比显示上数目少
- 特征关键点效果切换的实时性
- 支持bbox划区（上传图后在预览部分直接划就好）（发现检索耗时和图大小有关，同一张图划不同的bbox耗时不同）

## 尚未
- 直观看到做了什么，每一步都可以具象化


# 参数调试
## 1.TOP_N_PREFILTER 
- 100: 0.5859 (15min)
- 200: 0.6187 (20min)
- 300: 0.6274 (30min) ✅️
- 400: 0.6297 (44min)
- 暂定：300
## 1.1 评价
- 原理：好的结果可能排在 100 名之后，被截断了；候选变多时，正确的不容易被排掉
- 边际收益已经非常小，再增大到 500：预期提升 <0.1%，但计算时间会增加（RANSAC 处理更多候选图），也引入更多噪声（排名靠后的图片质量差）

## 2.对查询图做多尺度特征提取
提取 SIFT 时，除了原图，再对图像缩放 各提取一次，合并特征。相当于增加特征点数量，但不增加词汇量
- 单尺度：0.6297 (30min)
- 多尺度[1.0, 0.7, 1.4]：0.6324 (60min)
- 多尺度[1.0, 0.8]：0.6342 (40min) ✅️
- 多尺度[1.0, 0.7] / [1.0, 0.9] ：没试
- 多尺度[1.0, 0.8,1.2]: 0.6486 (58min) 【这个其实是我最后试的】
- 没选[1.0, 0.8,1.2]，因为考虑到时间增加 50%（40min → 60min），mAP 提升小，甚至可能下降（引入噪声）；不过可以试试
### 2.1 评价
- 现在是“建库单尺度 + 查询多尺度”，(因为我在自己这边跑不了50000的)；
- 还是推荐试试“建库多尺度 + 查询单尺度”，因为按道理：
  - 离线建库只做一次：花 90 分钟建库，之后永久受益
  - 查询更快：每次用户上传图片，不需要多尺度提取，用户体验好
  - mAP 同样高：数据库特征丰富，查询时单尺度也能匹配到
- 不过也可能：多尺度后特征点数量会增加 2-3 倍，可能影响内存和速度

## 3. 软分配
- 硬分配（即当前的）：0.6342 (40min)  ✅️
- top_k=3，sigma=1: 0.6205 (48min)
- top_k=2，sigma=0.6: 0.6298 (50min)

## 4.特征白化(没试)
原理：对 SIFT 描述子做 PCA 降维 + 白化，消除特征维度间的相关性，让每个维度等权重
### 4.1 评价
- 没尝试，因为要在 build_vocab.py 中训练白化矩阵，在聚类之前，先采样特征，计算 PCA 白化矩阵，我跑不了50000的，就没试

## 5.SIFT 参数调优 (没试)
SIFT 特征点：3000 → 5000
### 5.1 评价
- 没尝试
- 改完要重新跑这些
```bash
python scripts/feature_extract.py      # 重新提取特征
python scripts/build_vocab.py          # 重新训练词典（可选，建议）
python scripts/build_index.py          # 重建索引
python scripts/ransac_aqe_retrieval.py # 重新评测
```

## 6. AQE 权重
当前 AQE（平均）→ 加权AQE（按内点数加权）
- 当前 AQE（平均）：0.6342 (40min) ✅️
- 加权AQE：0.6193 (43min)
### 6.1 评价
针对现况，加权 AQE 和软分配都无效了，所以还是需要尝试 
- 更多词汇量（需要降采样来缓和内存）
- 特征白化
- SIFT 特征点增加

## 7.MIN_INLIERS_REQUIRED
- 10：0.6342 (40min) ✅️
- 15：0.6342 (40min) 
### 7.1 评价
- 没变化呀，但是你可以试试

## 8.top_k_expand
- 5: 0.6342 (40min) 
- 3: 0.6264 (37min)
- 7: 0.6346 (38min) ✅️
### 8.1 评价
- 其实也没什么变化

## 9.最终分数融合
- 排序只用内点数(当前): 0.6346 (38min) ✅️
- 内点数 + BoW 分数加权: 0.6346 (39min)

## 10. RANSAC_REPROJ_THRESHOLD
- 5.0: 0.6346 (38min) ✅️
- 3.0: 0.6346 (38min)
- 7.0: 0.6346 (38min)

# 综上
- 基本上都是有顺序的，比如“2.对查询图做多尺度特征提取”是在“1.TOP_N_PREFILTER”的基础上调参的；除了“多尺度[1.0, 0.8,1.2]: 0.6486 (58min)”是最后的尝试
- 优先试有效果的，有的可能没效果的有时间也试试吧
- 在第一个上打对钩，说明效果不太好
- 除了mAP,还要考虑用时和其他指标，综合考虑
